# **Transformers**

## **Setup**

In [22]:
import torch
import pandas as pd
import numpy as np
import json
import os
import re
from huggingface_hub import hf_hub_download
from sklearn.metrics import cohen_kappa_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
  print(f'GPU: {torch.cuda.get_device_name()}')

Device: cuda
GPU: Tesla T4


## **Download model and sample data from hugging face**

In [23]:
# transformer model
HF_REPO = "chimppy2k/asap-aes-transfer"

zs_ckpt_path = hf_hub_download(repo_id=HF_REPO, filename="zero_shot/stage_s_final.pt")
da_ckpt_path = hf_hub_download(repo_id=HF_REPO, filename="domain_adapted/stage_u_final.pt")

bin_edges_path = hf_hub_download(repo_id=HF_REPO, filename="bin_edges.json")
sample_data_path = hf_hub_download(repo_id=HF_REPO, filename="asap2_sample_50.csv")

with open(bin_edges_path) as f:
    bin_edges = np.array(json.load(f)['bin_edges'])
print(f'Bin edges (6 ordinal bins): {bin_edges}')

Bin edges (6 ordinal bins): [0.33333333 0.5        0.6        0.66666667 0.8       ]


## **Model Architecture**

Longformer-base-4096 \
**feature projection** linear-tanh-dropout \
**Regression head** \
**CORN ordinal head**

In [24]:
import torch.nn as nn
from transformers import LongformerModel, LongformerTokenizerFast
from peft import LoraConfig, get_peft_model


class CORNOrdinalHead(nn.Module):
    def __init__(self, in_features, num_classes=6):
        super().__init__()
        self.num_classes = num_classes
        self.fc = nn.Linear(in_features, num_classes - 1)

    def forward(self, x):
        return self.fc(x)

    def predict_labels(self, logits):
        probs = torch.sigmoid(logits)
        cumulative = torch.cumprod(probs, dim=1)
        return (cumulative > 0.5).sum(dim=1)


class AESModel(nn.Module):
  def __init__(self, hidden_dim=768, num_classes=6, dropout=0.1):
    super().__init__()
    self.longformer = LongformerModel.from_pretrained('allenai/longformer-base-4096')
    self.feature_proj = nn.Sequential(
      nn.Linear(hidden_dim, hidden_dim), nn.Tanh(), nn.Dropout(dropout))
    self.regression_head = nn.Linear(hidden_dim, 1)
    self.ordinal_head = CORNOrdinalHead(hidden_dim, num_classes)

  def predict(self, input_ids, attention_mask):
    global_attention_mask = torch.zeros_like(input_ids)
    global_attention_mask[:, 0] = 1
    outputs = self.longformer(
      input_ids=input_ids, attention_mask=attention_mask,
      global_attention_mask=global_attention_mask)
    features = self.feature_proj(outputs.last_hidden_state[:, 0, :])
    reg_score = torch.sigmoid(self.regression_head(features).squeeze(-1))
    ordinal_logits = self.ordinal_head(features)
    ordinal_labels = self.ordinal_head.predict_labels(ordinal_logits)
    return {'reg_score': reg_score, 'ordinal_labels': ordinal_labels}

class AESDAModel(AESModel):
  def __init__(self, hidden_dim=768, num_classes=6, dropout=0.1):
    super().__init__(hidden_dim, num_classes, dropout)
    self.grl = nn.Identity()
    self.domain_head = nn.Sequential(
      nn.Linear(hidden_dim, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 2))


print('Model classes defined.')

Model classes defined.


## **Load both zero-shot and domain-adapted models**

In [25]:
def load_model(model_class, ckpt_path, device):
  model = model_class()
  tokenizer = LongformerTokenizerFast.from_pretrained('allenai/longformer-base-4096')
  tokenizer.add_tokens(['<PARA>'])
  model.longformer.resize_token_embeddings(len(tokenizer))

  lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=['query', 'key', 'value'],
    lora_dropout=0.1, bias='none')
  model.longformer = get_peft_model(model.longformer, lora_config)

  state_dict = torch.load(ckpt_path, map_location=device)
  model.load_state_dict(state_dict, strict=False)
  model = model.to(device).eval()
  return model, tokenizer

print('Loading zero-shot model...')
zs_model, tokenizer = load_model(AESModel, zs_ckpt_path, device)
print('Loading domain-adapted model...')
da_model, _ = load_model(AESDAModel, da_ckpt_path, device)
print('Both models loaded')

Loading zero-shot model...


Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading domain-adapted model...


Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Both models loaded


## **Get Sample ASAP2 Essays ready**

In [26]:
ASAP2_PROMPTS = {
  '"A Cowboy Who Rode the Waves"': "Source-based: describe the narrator's perspective and experiences.",
  'Car-free cities': "Write about the advantages or disadvantages of limiting car usage in cities.",
  'Does the electoral college work?': "Write about whether the electoral college system is effective.",
  'Driverless cars': "Write about the development and implications of driverless car technology.",
  'Exploring Venus': "Write about the challenges and value of exploring the planet Venus.",
  'Facial action coding system': "Write about the value of technology that reads facial expressions.",
  'The Face on Mars': "Write about the landform on Mars known as the Face and what it represents.",
}

def preprocess(text):
  if not isinstance(text, str): return ''
  text = re.sub(r'\r\n', '\n', text)
  text = re.sub(r'\n{2,}', ' <PARA> ', text)
  text = re.sub(r'\n', ' ', text)
  return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv(sample_data_path)
df['text'] = df.apply(
    lambda r: preprocess(ASAP2_PROMPTS.get(r['prompt_name'], '')) + ' </s> ' + preprocess(r['full_text']),
    axis=1)

print(f'Loaded {len(df)} sample ASAP2 essays')
print(f'Prompts: {df["prompt_name"].unique().tolist()}')
print(f'Score distribution: {df["score"].value_counts().sort_index().to_dict()}')

Loaded 50 sample ASAP2 essays
Prompts: ['Exploring Venus']
Score distribution: {1: 7, 2: 22, 3: 17, 4: 4}


## **Run some predictions with zero-shot vs domain-adapted**

In [27]:
@torch.no_grad()
def predict_batch(model, texts, tokenizer, device, max_length=2048):
  enc = tokenizer(texts, max_length=max_length, padding='max_length',
                  truncation=True, return_tensors='pt')
  input_ids = enc['input_ids'].to(device)
  attention_mask = enc['attention_mask'].to(device)
  global_attention_mask = torch.zeros_like(input_ids)
  global_attention_mask[:, 0] = 1

  with torch.amp.autocast('cuda', dtype=torch.float16, enabled=device.type == 'cuda'):
    outputs = model.longformer(
      input_ids=input_ids, attention_mask=attention_mask,
      global_attention_mask=global_attention_mask)
    features = model.feature_proj(outputs.last_hidden_state[:, 0, :])
    reg_score = torch.sigmoid(model.regression_head(features).squeeze(-1))
    ordinal_logits = model.ordinal_head(features)
    probs = torch.sigmoid(ordinal_logits)
    ordinal_labels = (torch.cumprod(probs, dim=1) > 0.5).sum(dim=1)

  return reg_score.cpu().numpy(), ordinal_labels.cpu().numpy()

batch_size = 4
zs_reg, zs_ord, da_reg, da_ord = [], [], [], []

for i in range(0, len(df), batch_size):
  texts = df['text'].iloc[i:i+batch_size].tolist()

  reg, ord_ = predict_batch(zs_model, texts, tokenizer, device)
  zs_reg.extend(reg); zs_ord.extend(ord_)

  reg, ord_ = predict_batch(da_model, texts, tokenizer, device)
  da_reg.extend(reg); da_ord.extend(ord_)

df['zs_pred_reg'] = np.clip(np.round(np.array(zs_reg) * 5 + 1).astype(int), 1, 6)
df['zs_pred_ord'] = np.clip(np.array(zs_ord) + 1, 1, 6)
df['da_pred_reg'] = np.clip(np.round(np.array(da_reg) * 5 + 1).astype(int), 1, 6)
df['da_pred_ord'] = np.clip(np.array(da_ord) + 1, 1, 6)

print('Predictions complete!')

Predictions complete!


## **QWK Comparisons Results**

In [28]:
true = df['score'].astype(int).values

results = {
  'Zero-Shot (ordinal)': cohen_kappa_score(true, df['zs_pred_ord'], weights='quadratic'),
  'Zero-Shot (regression)': cohen_kappa_score(true, df['zs_pred_reg'], weights='quadratic'),
  'Domain-Adapted (ordinal)': cohen_kappa_score(true, df['da_pred_ord'], weights='quadratic'),
  'Domain-Adapted (regression)': cohen_kappa_score(true, df['da_pred_reg'], weights='quadratic'),
}

print('=' * 55)
print('QWK Results on ASAP2 Sample (50 essays)')
print('=' * 55)
for name, qwk in results.items():
  print(f'  {name:<30} QWK = {qwk:.4f}')
print()
print(f'  Improvement: {max(results["Domain-Adapted (ordinal)"], results["Domain-Adapted (regression)"]) - max(results["Zero-Shot (ordinal)"], results["Zero-Shot (regression)"]):.4f}')
# how lucky of me that the random 50 samples makes my model score much worse

QWK Results on ASAP2 Sample (50 essays)
  Zero-Shot (ordinal)            QWK = 0.0942
  Zero-Shot (regression)         QWK = 0.1038
  Domain-Adapted (ordinal)       QWK = 0.1592
  Domain-Adapted (regression)    QWK = 0.1745

  Improvement: 0.0706


# **Neural Networks**

##Setup


In [29]:
#import some libraries to be used

import pickle
import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

!pip install torchbnn

## Download Model and Sample Data from Hugging Face

In [30]:
#download models and data from hugging face

Hf_repo = "TheRealDestroyer/CS175Project"
mlp_path = hf_hub_download(repo_id=Hf_repo, filename="MLP_model")
bnn_path = hf_hub_download(repo_id=Hf_repo, filename="BNN_model")
inference_data_path = hf_hub_download(repo_id=Hf_repo, filename="inference_data.pkl")

## NN Declaration

In [31]:
%%writefile NN.py

#The parent class of MLP and BNN that trains, and test.

import torch
import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class NN:
    def calculate_grad(self, model):
        gradient_mag = 0
        for parameter in model.parameters():
            if parameter.grad is not None:
                gradient_mag += parameter.grad.data.norm(2).item() ** 2
        return gradient_mag ** 0.5

    def train_epoch(self, train_loader):
        loss_average = []

        y_pred = []
        y_true = []

        for batch_idx, (data, target) in enumerate(tqdm.tqdm(train_loader, desc="training", leave=True)):

            outputs = self.model(data.to(device=device)).squeeze()
            target = target.to(device=device).squeeze()

            y_pred.extend(outputs.detach().cpu().numpy().flatten())
            y_true.extend(target.detach().cpu().numpy())

            target = target[:, 0]

            loss = self.loss_function(outputs, target)
            loss_average.append(loss.item())

            self.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)

            gradient_mag = self.calculate_grad(self.model)

            if abs(gradient_mag) > 10 ** 3:
                print("Gradient exploding\n" + str(gradient_mag))

            if abs(gradient_mag) < 10 ** -3:
                print("Gradient vanishing\n" + str(gradient_mag))

            self.optimizer.step()

        return y_pred, y_true

    def test_model(self, test_loader):
        with torch.no_grad():

            loss_average = []

            y_pred = []
            y_true = []

            for batch_idx, (data, target) in enumerate(tqdm.tqdm(test_loader, desc="testing", leave=True)):
                outputs = self.model(data.to(device=device)).squeeze()
                target = target.to(device=device).squeeze()

                y_pred.extend(outputs.detach().cpu().numpy().flatten())
                y_true.extend(target.detach().cpu().numpy())

                target = target[:, 0]

                loss = self.loss_function(outputs, target)
                loss_average.append(loss.item())

            return y_pred, y_true

    def train_and_test_model(self, train_loader, test_loader, Asap2test_loader, num_epochs):
        y_pred_list_train = []
        y_true_list_train = []

        y_pred_list_val = []
        y_true_list_val = []

        y_pred_list_test = []
        y_true_list_test = []

        for epoch in range(num_epochs):
            print("Epoch: " + str(epoch))
            train_y_pred, train_y_true_all = self.train_epoch(train_loader)

            y_pred_list_train.append(train_y_pred)
            y_true_list_train.append(train_y_true_all)

            val_y_pred, val_y_true_all = self.test_model(test_loader)

            y_pred_list_val.append(val_y_pred)
            y_true_list_val.append(val_y_true_all)

            test_y_pred, test_y_true_all = self.test_model(Asap2test_loader)

            y_pred_list_test.append(test_y_pred)
            y_true_list_test.append(test_y_true_all)

        return y_pred_list_train, y_true_list_train, y_pred_list_val, y_true_list_val, y_pred_list_test, y_true_list_test

    def run_NN(self, train_loader, test_loader, Asap2test_loader, epochs):

        y_pred_list_train, y_true_list_train, y_pred_list_val, y_true_list_val, y_pred_list_test, y_true_list_test = self.train_and_test_model(train_loader, test_loader, Asap2test_loader, num_epochs=epochs)

        return y_pred_list_train, y_true_list_train, y_pred_list_val, y_true_list_val, y_pred_list_test, y_true_list_test


    def get_name(self):
        raise NotImplementedError

Overwriting NN.py


## MLP Declaration

In [32]:
%%writefile MLP.py

#The child class of NN, specifying what neural network it is

from NN import NN
import torch
class MLP(NN, torch.nn.Module):
    def __init__(self, in_features, out_features, cells):
        super(MLP, self).__init__()
        layers = []
        layers.append(torch.nn.Linear(in_features, cells[0]))
        layers.append(torch.nn.ReLU())
        for i in range(len(cells) - 1):
            layers.append(torch.nn.Linear(cells[i], cells[i + 1]))
            layers.append(torch.nn.ReLU())
        layers.append(torch.nn.Linear(cells[-1], out_features))
        layers.append(torch.nn.Sigmoid())
        self.model = torch.nn.Sequential(*layers)

        self.loss_function = torch.nn.MSELoss()

        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.001)

    def forward(self, x):
        return self.model(x)

    def get_name(self):
        return "MLP"

Overwriting MLP.py


## BNN Declaration

In [33]:
%%writefile BNN.py

#The child class of NN, specifying what neural network it is

from NN import NN
import torch
import torchbnn as bnn
class BNN(NN, torch.nn.Module):
    def __init__(self, in_features, out_features, cells):
        super(BNN, self).__init__()
        layers = []
        layers.append(bnn.modules.linear.BayesLinear(in_features = in_features, out_features = cells[0], prior_mu = 0, prior_sigma = 0.1))
        layers.append(torch.nn.ReLU())
        for i in range(len(cells) - 1):
            layers.append(bnn.modules.linear.BayesLinear(in_features = cells[i], out_features = cells[i + 1], prior_mu = 0, prior_sigma = 0.1))
            layers.append(torch.nn.ReLU())
        layers.append(bnn.modules.linear.BayesLinear(in_features = cells[-1], out_features = out_features, prior_mu = 0, prior_sigma = 0.1))
        layers.append(torch.nn.Sigmoid())
        self.model = torch.nn.Sequential(*layers)

        self.loss_function = torch.nn.MSELoss()

        self.optimizer = torch.optim.Adam(self.parameters(), lr=0.001)

    def forward(self, x):
        return self.model(x)

    def get_name(self):
        return "BNN"


Overwriting BNN.py


## Processing Functions

In [34]:
#these functions are for converting the scores back and calculating the val and test data

def create_loader(X_test, Y_test):
    test_dataset = TensorDataset(X_test, Y_test)

    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

    return test_loader

ASAP1_RATER_RANGES = {
    1: (1, 6),    # persuasive: effects of computers
    2: (1, 6),    # persuasive: censorship in libraries
    3: (0, 3),    # source-based: mood in memoir
    4: (0, 3),    # source-based: obstacles builders faced
    5: (0, 4),    # source-based: setting from narrator's perspective
    6: (0, 4),    # source-based: challenges of space travel
    7: (0, 12),   # narrative: patience
    8: (0, 30),   # narrative: laughter
}

def denormalize_asap1(norm_score: float, prompt_id: int) -> float:
    """Convert [0, 1] back to ASAP1 averaged rater scale."""
    lo, hi = ASAP1_RATER_RANGES[prompt_id]
    return norm_score * (hi - lo) + lo

ASAP2_SCORE_RANGE = (1, 6)

def denormalize_asap2(norm_score: float) -> float:
    """Convert [0, 1] back to ASAP2 scale (1-6)."""
    lo, hi = ASAP2_SCORE_RANGE
    return norm_score * (hi - lo) + lo

def denormalize_scores(y_pred_list, y_true_list, asap_version):
    y_pred_list_denorm = []
    y_true_list_denorm = []

    for i in range(len(y_pred_list)):
        prompt_id = y_true_list[i][1]

        if asap_version == 1:
            y_pred_list[i] = int(round(denormalize_asap1(y_pred_list[i], prompt_id)))
            y_pred_list_denorm.append(y_pred_list[i])

            y_true_list[i][0] = int(round(denormalize_asap1(y_true_list[i][0], prompt_id)))
            y_true_list_denorm.append(y_true_list[i])
        else:
            y_pred_list[i] = int(round(denormalize_asap2(y_pred_list[i])))
            y_pred_list_denorm.append(y_pred_list[i])

            y_true_list[i][0] = int(round(denormalize_asap2(y_true_list[i][0])))
            y_true_list_denorm.append(y_true_list[i])

    return y_pred_list_denorm, y_true_list_denorm

def inference_data_visualization(nn_name, y_pred_list_val, y_true_list_val, y_pred_list_test, y_true_list_test, plot):
    y_pred_val, y_true_val = denormalize_scores(y_pred_list_val, y_true_list_val, 1)
    y_true_val_true_only = np.array(y_true_val)[:, 0]

    val_qwk = (cohen_kappa_score(y_pred_val, y_true_val_true_only, weights='quadratic'))

    y_pred_test, y_true_test = denormalize_scores(y_pred_list_test, y_true_list_test, 2)
    y_true_test_true_only = np.array(y_true_test)[:, 0]

    test_qwk = (cohen_kappa_score(y_pred_test, y_true_test_true_only, weights='quadratic'))

    if plot:
        plt.title(nn_name + " Val and Test QWK for Inference")
        plt.bar("val", val_qwk, label="Val QWK")
        plt.text(0, val_qwk / 2, val_qwk, ha='center')
        plt.bar("test", test_qwk, label="Testing QWK")
        plt.text(1, test_qwk / 2, test_qwk, ha='center')
        plt.ylabel('QWK')
        plt.xlabel('Epoch')
        plt.legend()

        plt.show()

    print()
    print(nn_name + " Val and Test QWK for Inference:")
    print("Val QWK: " + str(val_qwk))
    print("Test QWK: " + str(test_qwk))
    print()


## Inference

In [35]:
import MLP
import BNN

#puts it all together resulting in val and test qwk randomly sampled from a small subset of our data

data = pickle.load(open(inference_data_path, 'rb'))

val = data["val"]
test = data["test"]

random_number = np.random.randint(0, 200)

val_loader = create_loader(val["X_val"][random_number:random_number+250], val["Y_val"][random_number:random_number+250])
test_loader = create_loader(test["X_test"][random_number:random_number+250], test["Y_test"][random_number:random_number+250])

mlp = torch.load(mlp_path, weights_only=False)
mlp.eval()

y_pred_list_val, y_true_list_val = mlp.test_model(val_loader)
y_pred_list_test, y_true_list_test = mlp.test_model(test_loader)

plot = False

inference_data_visualization(mlp.get_name(), y_pred_list_val, y_true_list_val, y_pred_list_test, y_true_list_test, plot)

bnn = torch.load(bnn_path, weights_only=False)
bnn.eval()

y_pred_list_val, y_true_list_val = bnn.test_model(val_loader)
y_pred_list_test, y_true_list_test = bnn.test_model(test_loader)

inference_data_visualization(bnn.get_name(), y_pred_list_val, y_true_list_val, y_pred_list_test, y_true_list_test, plot)

testing: 100%|██████████| 8/8 [00:00<00:00, 297.63it/s]



MLP Val and Test QWK for Inference:
Val QWK: 0.9669660752728082
Test QWK: 0.31010236946758796



testing: 100%|██████████| 8/8 [00:00<00:00, 128.95it/s]


BNN Val and Test QWK for Inference:
Val QWK: 0.9485132492571912
Test QWK: 0.3336846860412588



# Ridge Regression (Linear Regression)

## Setup


In [36]:
%pip install -U scikit-learn
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, cohen_kappa_score
from huggingface_hub import hf_hub_download
import joblib
import pickle
import numpy as np
import os

## Download Model and Samples

In [37]:
# ridge regression model
HF1_REPO = "Ueltz/asap-aes-transfer"

# download ridge regression models
# note that the models are directly downloaded and then loaded (so no training is necessary)
prompt_ridge_path = hf_hub_download(repo_id=HF1_REPO, filename="best_prompt_ridge_model.joblib")
noprompt_ridge_path = hf_hub_download(repo_id=HF1_REPO, filename="best_noprompt_ridge_model.joblib")
data_path = hf_hub_download(repo_id=HF1_REPO, filename="embeddings_no_prompt_with_demographics.pkl")
data_path_prompt = hf_hub_download(repo_id=HF1_REPO, filename="embeddings.pkl")
asap1_train_labels_path = hf_hub_download(repo_id=HF1_REPO, filename="train_prompt_ids.pkl")
asap1_val_labels_path = hf_hub_download(repo_id=HF1_REPO, filename="val_prompt_ids.pkl")

## Processing Functions

In [38]:
def embed_and_pickle():
    with open(data_path, 'rb') as f:
        data = pickle.load(f)

        train = data["asap1_train"]
        val = data["asap1_val"]
        test = data["asap2"]

        X_train = train["X_train"]

        Y_train = train["Y_train"]
        X_val = val["X_val"]
        Y_val = val["Y_val"]

        X_test = test["X_test"]
        Y_test = test["Y_test"]

        return X_train, Y_train, X_val, Y_val, X_test, Y_test

def evaluate_model(y_true_norm, y_pred_norm, dataset_name, y_true_raw=None, y_pred_raw=None):
    """Compute metrics using normalized scores, except kappa uses raw values.

    Parameters:
    - y_true_norm, y_pred_norm: normalized [0,1] arrays for MSE/MAE/accuracy
    - y_true_raw, y_pred_raw: denormalized arrays (same length) for quadratic kappa
    """
    mse = mean_squared_error(y_true_norm, y_pred_norm)
    mae = mean_absolute_error(y_true_norm, y_pred_norm)
    # round normalized for accuracy
    y_true_int = np.rint(y_true_norm)
    y_pred_int = np.rint(y_pred_norm)


    # kappa source: raw if provided else rounded normalized
    if y_true_raw is not None and y_pred_raw is not None:
        y_true_kappa = np.rint(y_true_raw)
        y_pred_kappa = np.rint(y_pred_raw)
    else:
        y_true_kappa = y_true_int
        y_pred_kappa = y_pred_int
    acc = accuracy_score(y_true_kappa, y_pred_kappa)
    kappa = cohen_kappa_score(y_true_kappa, y_pred_kappa, weights='quadratic')

    print(f"{dataset_name}:")
    print(f"  MSE: {mse:.4f}")
    print(f"  MAE: {mae:.4f}")
    print(f"  Accuracy: {acc:.4f} (rounded)")
    print(f"  Quadratic Weighted Kappa: {kappa:.4f}")
    print()


ASAP1_RATER_RANGES = {
    1: (1, 6),    # persuasive: effects of computers
    2: (1, 6),    # persuasive: censorship in libraries
    3: (0, 3),    # source-based: mood in memoir
    4: (0, 3),    # source-based: obstacles builders faced
    5: (0, 4),    # source-based: setting from narrator's perspective
    6: (0, 4),    # source-based: challenges of space travel
    7: (0, 12),   # narrative: patience
    8: (0, 30),   # narrative: laughter
}

# ASAP2: uniform 1–6 scoring
ASAP2_SCORE_RANGE = (1, 6)

DEFAULT_VAL_FRACTION = 0.15
DEFAULT_SEED = 42

def normalize_asap1(avg_score: float, prompt_id: int) -> float:
    """Normalize averaged ASAP1 rater score to [0, 1]."""
    lo, hi = ASAP1_RATER_RANGES[prompt_id]
    if hi == lo:
        return 0.5
    return (avg_score - lo) / (hi - lo)


def denormalize_asap1(norm_score: float, prompt_id: int) -> float:
    """Convert [0, 1] back to ASAP1 averaged rater scale."""
    lo, hi = ASAP1_RATER_RANGES[prompt_id]
    return round(norm_score * (hi - lo) + lo) # for accurate integer conversion (prefer round up)


def normalize_asap2(score: float) -> float:
    """Normalize ASAP2 score (1-6) to [0, 1]."""
    lo, hi = ASAP2_SCORE_RANGE
    return (score - lo) / (hi - lo)


def denormalize_asap2(norm_score: float) -> float:
    """Convert [0, 1] back to ASAP2 scale (1-6)."""
    lo, hi = ASAP2_SCORE_RANGE
    return round(norm_score * (hi - lo) + lo) # for accurate integer conversion (prefer round up)


## Load Models & Data (No Prompt Embeddings)

In [39]:
# load models
prompt_model = joblib.load(prompt_ridge_path)
noprompt_model = joblib.load(noprompt_ridge_path)

X_train, Y_train, X_val, Y_val, X_test, Y_test = embed_and_pickle()

X_train = np.asarray(X_train)
y_train = np.asarray(Y_train[:, 0])

X_val = np.asarray(X_val)
y_val = np.asarray(Y_val[:, 0])

X_asap2 = np.asarray(X_test)
y_asap2 = np.asarray(Y_test[:, 0])

train_prompt_ids = np.asarray(Y_train[:, 1])
val_prompt_ids = np.asarray(Y_val[:, 1])
asap2_prompt_ids = np.asarray(Y_test[:, 1])

y_train_true_raw = np.array([denormalize_asap1(n, pid)
        for n, pid in zip(np.array(Y_train[:,0]), train_prompt_ids)])

y_val_true_raw = np.array([denormalize_asap1(n, pid)
  for n, pid in zip(np.array(Y_val[:,0]), val_prompt_ids)])

y_asap2_true_raw = np.array([denormalize_asap2(n)
  for n in np.array(Y_test[:,0])])

def eval_model(model):
  y_train_pred = model.predict(X_train)
  y_val_pred = model.predict(X_val)
  y_asap2_pred = model.predict(X_asap2)

  y_train_pred_raw = np.array([
    denormalize_asap1(n, pid)
    for n, pid in zip(y_train_pred, train_prompt_ids)
  ])

  y_val_pred_raw = np.array([
    denormalize_asap1(n, pid)
    for n, pid in zip(y_val_pred, val_prompt_ids)
  ])

  y_asap2_pred_raw = np.array([
    denormalize_asap2(n) for n in y_asap2_pred
  ])

  evaluate_model(y_train, y_train_pred, "Training Set", y_train_true_raw, y_train_pred_raw)
  evaluate_model(y_val, y_val_pred, "Validation Set", y_val_true_raw, y_val_pred_raw)
  evaluate_model(y_asap2, y_asap2_pred, "ASAP 2.0 Set", y_asap2_true_raw, y_asap2_pred_raw)

## Evaluate models (On essays without prompts)

In [40]:
print("=" * 60 + "Evaluating prompt-based Ridge Model" + "=" * 60)
eval_model(prompt_model)
print()
print("=" * 60 + "Evaluating no-prompt-based Ridge Model" + "=" * 60)
eval_model(noprompt_model)

# for some reason, the results for SPECIFICALLY the prompt-based model differ from the results that were outputted on my end...

============================================================Evaluating prompt-based Ridge Model============================================================
Training Set:
  MSE: 0.0306
  MAE: 0.1358
  Accuracy: 0.4602 (rounded)
  Quadratic Weighted Kappa: 0.9237

Validation Set:
  MSE: 0.0343
  MAE: 0.1439
  Accuracy: 0.4499 (rounded)
  Quadratic Weighted Kappa: 0.9165

ASAP 2.0 Set:
  MSE: 0.0702
  MAE: 0.2149
  Accuracy: 0.2787 (rounded)
  Quadratic Weighted Kappa: 0.2362


============================================================Evaluating no-prompt-based Ridge Model============================================================
Training Set:
  MSE: 0.0171
  MAE: 0.1020
  Accuracy: 0.5551 (rounded)
  Quadratic Weighted Kappa: 0.9662

Validation Set:
  MSE: 0.0212
  MAE: 0.1126
  Accuracy: 0.5239 (rounded)
  Quadratic Weighted Kappa: 0.9588

ASAP 2.0 Set:
  MSE: 0.0583
  MAE: 0.1958
  Accuracy: 0.3056 (rounded)
  Quadratic Weighted Kappa: 0.2482



## Load Data (With Prompt Embeddings)

In [41]:
def load_tokenized_data(cache_path=data_path_prompt):
    if os.path.exists(cache_path):
        with open(cache_path, 'rb') as f:
            tokenized = pickle.load(f)
        return tokenized
    return None
tokenized = load_tokenized_data()

X_train = np.array(tokenized['asap1_train']['embeddings'])
y_train = np.array(tokenized['asap1_train']['scores'])

X_val = np.array(tokenized['asap1_val']['embeddings'])
y_val = np.array(tokenized['asap1_val']['scores'])

X_asap2 = np.array(tokenized['asap2']['embeddings'])
y_asap2 = np.array(tokenized['asap2']['scores'])

with open(asap1_train_labels_path, 'rb') as f:
  train_prompt_ids = pickle.load(f)

with open(asap1_val_labels_path, 'rb') as f:
  val_prompt_ids = pickle.load(f)

y_train_true_raw = np.array([denormalize_asap1(n, pid)
      for n, pid in zip(y_train, train_prompt_ids)])
y_val_true_raw = np.array([denormalize_asap1(n, pid)
      for n, pid in zip(y_val, train_prompt_ids)])
y_asap2_true_raw = np.array([
      denormalize_asap2(n) for n in y_asap2
  ])

def eval_model_prompt(model):
  y_train_pred = model.predict(X_train)
  y_val_pred = model.predict(X_val)
  y_asap2_pred = model.predict(X_asap2)

  y_train_pred_raw = np.array([
      denormalize_asap1(n, pid)
      for n, pid in zip(y_train_pred, train_prompt_ids)
  ])

  y_val_pred_raw = np.array([
      denormalize_asap1(n, pid)
      for n, pid in zip(y_val_pred, val_prompt_ids)
  ])

  y_asap2_pred_raw = np.array([
      denormalize_asap2(n) for n in y_asap2_pred
  ])

  evaluate_model(y_train, y_train_pred, "Training Set", y_train_true_raw, y_train_pred_raw)
  evaluate_model(y_val, y_val_pred, "Validation Set", y_val_true_raw, y_val_pred_raw)
  evaluate_model(y_asap2, y_asap2_pred, "ASAP 2.0 Set", y_asap2_true_raw, y_asap2_pred_raw)

## Evaluate models (On essays with prompts)

In [42]:
print("=" * 60 + "Evaluating prompt-based Ridge Model" + "=" * 60)
eval_model(prompt_model)
print()
print("=" * 60 + "Evaluating no-prompt-based Ridge Model" + "=" * 60)
eval_model_prompt(noprompt_model)

============================================================Evaluating prompt-based Ridge Model============================================================
Training Set:
  MSE: 0.0147
  MAE: 0.0937
  Accuracy: 0.5828 (rounded)
  Quadratic Weighted Kappa: 0.9747

Validation Set:
  MSE: 0.0183
  MAE: 0.1034
  Accuracy: 0.2265 (rounded)
  Quadratic Weighted Kappa: 0.1356

ASAP 2.0 Set:
  MSE: 0.0853
  MAE: 0.2435
  Accuracy: 0.2254 (rounded)
  Quadratic Weighted Kappa: 0.1812


============================================================Evaluating no-prompt-based Ridge Model============================================================
Training Set:
  MSE: 0.0191
  MAE: 0.1086
  Accuracy: 0.5327 (rounded)
  Quadratic Weighted Kappa: 0.9660

Validation Set:
  MSE: 0.0213
  MAE: 0.1135
  Accuracy: 0.2224 (rounded)
  Quadratic Weighted Kappa: 0.1206

ASAP 2.0 Set:
  MSE: 0.0737
  MAE: 0.2262
  Accuracy: 0.2432 (rounded)
  Quadratic Weighted Kappa: 0.1624

